# P0c: calibration-only historical-utility predictability screen

Reads the integrity-checked Chronos-2 and TimesFM-3 P0b artifacts from Google Drive. For each later calibration origin, it scores a candidate policy using only the same policy's strictly earlier calibration utilities, then computes task-macro utility-sign AUROC. It performs **no model inference** and never instantiates sealed evaluation origins. Results are `screening_only`.

Use a **CPU runtime**. This analysis should finish in minutes.

In [ ]:
import importlib
import subprocess
import sys
from pathlib import Path

REPO = Path('/content/covariate-safe-tsfm')
if not REPO.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/FlyMe2star/covariate-safe-tsfm.git', str(REPO)],
        check=True,
    )
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)],
    check=True,
)
SOURCE_ROOT = str(REPO / 'src')
if SOURCE_ROOT not in sys.path:
    sys.path.insert(0, SOURCE_ROOT)
importlib.invalidate_caches()
covsafe = importlib.import_module('covsafe')
print('Repository ready:', REPO)
print('Git commit:', subprocess.check_output(
    ['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True
).strip())

In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
PRIVATE_ROOT = Path(
    '/content/drive/MyDrive/covariate-safe-tsfm/private_manifests'
)
P0B_ROOT = PRIVATE_ROOT / 'p0b'
P0C_ROOT = PRIVATE_ROOT / 'p0c'
assert P0B_ROOT.exists(), 'P0b artifacts are missing from Google Drive.'
P0C_ROOT.mkdir(parents=True, exist_ok=True)
print('P0b input root:', P0B_ROOT)
print('P0c durable root:', P0C_ROOT)

In [ ]:
import json

from covsafe.p0c import EXPECTED_P0C_CONFIG_HASH, run_p0c

print('Frozen P0c config hash:', EXPECTED_P0C_CONFIG_HASH)
report = run_p0c(REPO, P0B_ROOT, P0C_ROOT)
print(json.dumps(report, indent=2, ensure_ascii=False, default=str))

## Return artifact

Send the final JSON containing `backbones` and `cross_backbone_gate`. Do not open or run any sealed-evaluation notebook until this report has been reviewed.